# UPI Fraud Rings: full pipeline on Colab

Runs `run_all.py` (raw CSVs to submission) and an extra XGBoost model on the GPU, then blends the two.

Before running:
1. Put the five raw CSVs in Drive at `MyDrive/upi-fraud/data/raw/`.
2. In the Colab **Secrets** panel add `GH_TOKEN`, a GitHub token that can read the private repo.
3. Pick a GPU runtime. **Disconnect the runtime when the last cell finishes**: compute units burn while it stays connected.

In [ ]:
!nvidia-smi -L; nproc; free -g | head -2

In [ ]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
REPO = 'zahy294/upi-fraud-ml'   # change the branch below if the code lives on another one
BRANCH = 'modelling'
!rm -rf /content/upi-fraud-ml && git clone -q -b $BRANCH https://$GH_TOKEN@github.com/$REPO.git /content/upi-fraud-ml
%cd /content/upi-fraud-ml
!git log -1 --oneline
!pip -q install lightgbm==4.7.0 xgboost==3.2.0 pyarrow scikit-learn pandas matplotlib scipy
!mkdir -p data && ln -sfn /content/drive/MyDrive/upi-fraud/data/raw data/raw && ls -la data/raw/

In [ ]:
# Full pipeline with LightGBM on the CPU cores. About 25-40 minutes depending on the core count.
!python run_all.py --seeds 5 --name final_lgb

In [ ]:
# Second model family on the GPU, same features and cleaned labels, then a blend of the two.
FEATS = 'features_v1.parquet,features_v2.parquet,features_v3.parquet,features_v4.parquet,features_v5.parquet'
!python -I src/upi/run_oof.py . oof_clean_xgb --features $FEATS --labels models/oof_raw/y_clean.npy --trees 300 --seeds 5 --model xgb --device cuda --test
!python -I src/upi/make_submission.py . oof_clean --labels models/oof_raw/y_clean.npy --blend oof_clean,oof_clean_xgb --name final_blend

In [ ]:
# Keep the outputs: submissions, probabilities, band edges and the package versions used.
!mkdir -p /content/drive/MyDrive/upi-fraud/runs/colab && cp submissions/final_*.csv data/processed/band_edges.json /content/drive/MyDrive/upi-fraud/runs/colab/
!cp -r models/oof_clean models/oof_clean_xgb /content/drive/MyDrive/upi-fraud/runs/colab/ && pip freeze > /content/drive/MyDrive/upi-fraud/runs/colab/pip_freeze.txt
!ls -la /content/drive/MyDrive/upi-fraud/runs/colab/